In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
import torchvision.models as models
from torchvision import transforms
import numpy as np
import random
from tqdm import tqdm
from utils.dataloader import *
import random
import matplotlib.pyplot as plt
import pprint

Currect path: /Users/nathan/Library/Mobile Documents/com~apple~CloudDocs/1-MIT/Spring 2026/6.S058/6.s058-Project


In [6]:
DATASET_PATH = '/Users/nathan/Library/Mobile Documents/com~apple~CloudDocs/1-MIT/Spring 2026/6.S058/6.s058-Project'
manga_name_list = get_book_list(DATASET_PATH)

In [7]:
def jitter_box(position, page_width, page_height, noise=15):
    x_min, y_min, x_max, y_max = position
    x_min += random.randint(-noise, noise)
    y_min += random.randint(-noise, noise)
    x_max += random.randint(-noise, noise)
    y_max += random.randint(-noise, noise)
    
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(page_width, x_max)
    y_max = min(page_height, y_max)
    
    x_max = max(x_min + 1, x_max)
    y_max = max(y_min + 1, y_max)
    
    return (x_min, y_min, x_max, y_max)

to_tensor = transforms.ToTensor()

class Manga109Dataset(Dataset):
    def __init__(self, dataset_path, manga_name, transform=jitter_box):
        self.dataset_path = dataset_path
        self.manga_name = manga_name
        self.transform = transform
        
        annotations = annotation_loader(dataset_path, manga_name)
        self.page_width = int(annotations['book']['pages']['page'][0]['@width'])
        self.page_height = int(annotations['book']['pages']['page'][0]['@height'])

        self.char_ids = []
        self.char_faces = {}
        
        for char_id in get_character_list(annotations)[0]:
            faces_info, _ = get_crops_info_char(annotations, char_id)
            if len(faces_info) >= 2:
                self.char_ids.append(char_id)
                self.char_faces[char_id] = faces_info

    def __len__(self):
        return len(self.char_ids)

    def __getitem__(self, index):
        char_id = self.char_ids[index]
        faces_info = self.char_faces[char_id]
        
        crop_i, crop_j = random.sample(faces_info, 2)
        
        if self.transform:
            pos_i = jitter_box(crop_i['position'], self.page_width, self.page_height)
            pos_j = jitter_box(crop_j['position'], self.page_width, self.page_height)
        else:
            pos_i = crop_i['position']
            pos_j = crop_j['position']
        
        xi = to_tensor(retrieve_page(self.dataset_path, self.manga_name, crop_i['@index'], pos_i, target_size=(224, 224)))
        xj = to_tensor(retrieve_page(self.dataset_path, self.manga_name, crop_j['@index'], pos_j, target_size=(224, 224)))

        return xi, xj

In [9]:
class SimCLRModel(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        base_model = models.resnet18(weights=None)
        num_ftrs = base_model.fc.in_features # use only fully connected layer (no final classification head)
        base_model.fc = nn.Identity()
        self.encoder = base_model
        self.projection_head = nn.Sequential( # project to embedding space
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return z

In [10]:
def nt_xent_loss(z_i, z_j, temperature=0.5):
    z = torch.cat([z_i, z_j], dim=0)
    z = F.normalize(z, dim=1)

    similarity = torch.matmul(z, z.T)
    N = z_i.shape[0]

    mask = (~torch.eye(2*N, dtype=bool)).to(z.device)
    sim = similarity / temperature
    exp_sim = torch.exp(sim) * mask

    positive_sim = torch.exp(F.cosine_similarity(z_i, z_j) / temperature)
    positives = torch.cat([positive_sim, positive_sim], dim=0)

    denominator = exp_sim.sum(dim=1)
    loss = -torch.log(positives / denominator)
    return loss.mean()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SimCLRModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

for manga_name in manga_name_list:
    print(manga_name)
    contrastive_dataset = Manga109Dataset(DATASET_PATH, manga_name)
    train_loader = DataLoader(contrastive_dataset, batch_size=32, shuffle=True, num_workers=0)

    for epoch in range(10):
        model.train()
        total_loss = 0
        for x_i, x_j in tqdm(train_loader):
            x_i, x_j = x_i.to(device), x_j.to(device)
            z_i = model(x_i)
            z_j = model(x_j)

            loss = nt_xent_loss(z_i, z_j)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"{manga_name} | Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
# Freeze encoder and train linear layer ontop to see how good the learned encoder is

for param in model.encoder.parameters():
    param.requires_grad = False

classifier = nn.Linear(512, 10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

def get_features_and_labels(loader):
    features, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            h = model.encoder(x) # use our trained model enncoder
            features.append(h.cpu())
            labels.append(y)
    return torch.cat(features), torch.cat(labels)